# Workshop 2 · KPI Layer + BI Readiness

**Goal:** Add the KPI layer on top of the star schema from Workshop 1. Build 4 new Gold objects — then connect Power BI. Run this at the start of Day 2.

In [ ]:
%run ../../setup/00_setup

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("SILVER",  SILVER),
    ("GOLD",    GOLD),
], ["Variable", "Value"]))

In [ ]:
prereqs = [
    f"{GOLD}.fact_sales_dashboard",
    f"{SILVER}.customers",
    f"{SILVER}.orders",
    f"{SILVER}.order_items",
]

In [ ]:
missing = [t for t in prereqs if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing prerequisite objects:")
    for t in missing: print(" -", t)
    raise Exception("Complete Workshop 1 first (gold.fact_sales_dashboard must exist).")
print("[OK] Pre-check passed — all prerequisites exist.")

## Task 1 — KPI Definitions & kpi_daily

| KPI | Definition | Filter |
|---|---|---|
| revenue | SUM(line_revenue) | is_completed = true |
| gross_margin | SUM(line_margin) | is_completed = true |
| completed_orders | COUNT(DISTINCT order_id) | is_completed = true |
| return_rate_pct | returned_orders / all_orders × 100 | all status |

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.kpi_daily AS
SELECT
  order_date,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END)  AS completed_orders,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin  ELSE 0 END), 2) AS gross_margin,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN status = 'Returned' THEN order_id END)
          / NULLIF(COUNT(DISTINCT order_id), 0), 2
  ) AS return_rate_pct
FROM gold.fact_sales_dashboard
GROUP BY order_date

In [ ]:
%sql
SELECT * FROM gold.kpi_daily ORDER BY order_date DESC LIMIT 10

## Task 2 — kpi_monthly

Roll up kpi_daily to monthly grain using dim_date.year_month. This is the KPI card source for Power BI.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.kpi_monthly AS
SELECT
  d.year_month,
  SUM(k.completed_orders)               AS completed_orders,
  ROUND(SUM(k.revenue), 2)              AS revenue,
  ROUND(SUM(k.gross_margin), 2)         AS gross_margin,
  ROUND(AVG(k.return_rate_pct), 2)      AS return_rate_pct
FROM gold.kpi_daily k
JOIN gold.dim_date d ON k.order_date = d.date_key
GROUP BY d.year_month
ORDER BY d.year_month

In [ ]:
%sql
SELECT * FROM gold.kpi_monthly ORDER BY year_month DESC LIMIT 6

## Task 3 — fact_sales_dashboard_monthly

Pre-aggregated monthly table: one row per month × customer_region × category × channel. Summary page source for Power BI — much faster to query than the detail grain table.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales_dashboard_monthly AS
SELECT
  year_month,
  customer_region,
  category,
  channel,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END)          AS completed_orders,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin  ELSE 0 END), 2) AS gross_margin
FROM gold.fact_sales_dashboard
GROUP BY year_month, customer_region, category, channel

In [ ]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT year_month) AS months
FROM gold.fact_sales_dashboard_monthly

## Task 4 — v_fact_sales_incremental

Bounded view: last 24 months of detail data. Used for Power BI incremental refresh (drill-through pages) — avoids scanning full history on every import.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.v_fact_sales_incremental AS
SELECT *
FROM gold.fact_sales_dashboard
WHERE order_date >= ADD_MONTHS(CURRENT_DATE(), -24)

In [ ]:
%sql
SELECT
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date,
  COUNT(*)        AS rows
FROM gold.v_fact_sales_incremental

## Task 5 — Reconciliation

Verify that kpi_daily rolled up to months matches kpi_monthly exactly.

In [ ]:
%sql
WITH dr AS (SELECT d.year_month, ROUND(SUM(k.revenue),2) AS rev_daily
  FROM gold.kpi_daily k JOIN gold.dim_date d ON k.order_date=d.date_key
  GROUP BY d.year_month)
SELECT m.year_month, m.revenue AS revenue_kpi_monthly,
       dr.rev_daily, ABS(m.revenue-dr.rev_daily) AS diff
FROM gold.kpi_monthly m JOIN dr ON m.year_month=dr.year_month ORDER BY diff DESC LIMIT 5

In [ ]:
_q = ("WITH dr AS (SELECT d.year_month, SUM(k.revenue) AS rev FROM gold.kpi_daily k "
      "JOIN gold.dim_date d ON k.order_date=d.date_key GROUP BY d.year_month) "
      "SELECT MAX(ABS(m.revenue-dr.rev)) AS max_diff FROM gold.kpi_monthly m "
      "JOIN dr ON m.year_month=dr.year_month")
result = spark.sql(_q).collect()[0]

In [ ]:
max_diff = result["max_diff"] or 0
assert max_diff < 0.01, f"Reconciliation FAILED: max diff = {max_diff}"
print(f"[OK] Reconciliation passed — max diff = {max_diff}")

## Task 6 — BI Readiness Self-Check

In [ ]:
%sql
SHOW TABLES IN gold

In [ ]:
_objs = ["dim_date", "dim_customer", "dim_product", "fact_sales",
         "fact_sales_dashboard", "kpi_daily", "kpi_monthly",
         "fact_sales_dashboard_monthly", "v_fact_sales_incremental"]
required = [f"{GOLD}.{o}" for o in _objs]

In [ ]:
missing = [t for t in required if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing Gold objects:")
    for t in missing: print(" -", t)
    raise Exception("Workshop 2 incomplete.")
print("[OK] All 9 Gold objects exist!")

Your Kimball model is complete. Connect Power BI to these objects:
- `gold.fact_sales_dashboard_monthly` → summary page (Import mode, small table)
- `gold.v_fact_sales_incremental` → drill-through page (last 24 months)
- `gold.kpi_monthly` → KPI cards
- `gold.dim_customer`, `gold.dim_date`, `gold.dim_product` → slicers

Your catalog: see CATALOG variable above (e.g., retailhub_jan_kowalski)